<a href="https://colab.research.google.com/github/AljawharahAlqabbani/SDAIA-projects/blob/main/Copy_of_Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Ingestion & Contract

In [ ]:
import pandas as pd
file_path = '/content/online_retail.csv'

df = pd.read_csv(file_path)
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


# Validation & Quarantine

In [ ]:
from pydantic import BaseModel, ValidationError, Field
import pandas as pd
import numpy as np

# 1. تعريف عقد البيانات (Data Contract) باستخدام Pydantic
class RetailRecord(BaseModel):
    InvoiceNo: str
    StockCode: str
    Description: str = None
    Quantity: int = Field(..., gt=0) # يجب أن تكون الكمية أكبر من الصفر
    InvoiceDate: str
    UnitPrice: float = Field(..., gt=0.0) # يجب أن يكون السعر أكبر من الصفر
    CustomerID: float = None
    Country: str

# قراءة عينة من البيانات للتحقق
df = pd.read_csv('/content/online_retail.csv')

valid_records = []
quarantine_records = []

# 2. عملية فحص الحدود (Ingestion Boundary Validation)
for index, row in df.iterrows():
    # تحويل السطر إلى قاموس
    row_dict = row.to_dict()

    # معالجة القيم المفقودة البسيطة لكي تتوافق مع Pydantic
    for k, v in row_dict.items():
        if pd.isna(v):
            row_dict[k] = None

    try:
        # محاولة التحقق عبر Pydantic Schema
        record = RetailRecord(**row_dict)
        valid_records.append(record.dict())
    except ValidationError as e:
        # إذا فشل التحقق، يتم إرساله لمنطقة الحجر الصحي (Quarantine Zone) مع سبب الرفض
        error_info = row_dict.copy()
        error_info['rejection_reason'] = str(e)
        quarantine_records.append(error_info)

# 3. طباعة نتائج الفحص
print(f"عدد السجلات السليمة (Valid Records): {len(valid_records)}")
print(f"عدد السجلات المرفوضة في الحجر الصحي (Quarantine Records): {len(quarantine_records)}")

# تحويل السجلات السليمة إلى DataFrame جديد للاستخدام في الخطوة القادمة
df_valid = pd.DataFrame(valid_records)
df_quarantine = pd.DataFrame(quarantine_records)

/tmp/ipykernel_2142/3060264858.py:35: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  valid_records.append(record.dict())


عدد السجلات السليمة (Valid Records): 397884
عدد السجلات المرفوضة في الحجر الصحي (Quarantine Records): 144025


# Delta Lakehouse Layers

In [ ]:
# 1. تثبيت مكتبة deltalake الداعمة لعمليات دلتا مباشرة
!pip install deltalake

from deltalake import write_deltalake
import pandas as pd

# تحويل البيانات السليمة إلى جدول Pandas DataFrame (إذا لم تكن موجودة)
# df_valid جاهزة لديك من الخطوة السابقة

# 2. حفظ طبقة Bronze
write_deltalake("/tmp/delta/bronze_retail", df_valid, mode="overwrite")
print("تم حفظ البيانات بنجاح في طبقة Bronze الخام.")

# 3. حفظ طبقة Silver (المنظفة)
write_deltalake("/tmp/delta/silver_retail", df_valid, mode="overwrite")
print("تم حفظ البيانات بنجاح في طبقة Silver المنظفة.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 107.5 MB/s eta 0:00:00
تم حفظ البيانات بنجاح في طبقة Bronze الخام.
تم حفظ البيانات بنجاح في طبقة Silver المنظفة.


In [ ]:
from deltalake import write_deltalake
import pandas as pd

# قراءة بيانات طبقة Silver التي حفظناها سابقاً
df_silver = pd.read_parquet("/tmp/delta/silver_retail") # أو قراءتها عبر deltalake

# بناء تجميع حقيقي لطبقة Gold (مثلاً: حساب إجمالي المبيعات والكميات لكل منتج StockCode)
df_gold = df_silver.groupby(['StockCode', 'Country']).agg(
    Total_Quantity=('Quantity', 'sum'),
    Total_Sales=('UnitPrice', lambda x: (x * df_silver.loc[x.index, 'Quantity']).sum()),
    Average_Price=('UnitPrice', 'mean'),
    Transaction_Count=('InvoiceNo', 'count')
).reset_index()

# حفظ طبقة Gold بصيغة Delta Lake
write_deltalake("/tmp/delta/gold_retail", df_gold, mode="overwrite")

print("تم بنجاح بناء وحفظ طبقة Gold التجميعية!")
df_gold.head()

تم بنجاح بناء وحفظ طبقة Gold التجميعية!


,StockCode,Country,Total_Quantity,Total_Sales,Average_Price,Transaction_Count
0,10002,EIRE,12,10.20,0.85,1
1,10002,France,372,316.20,0.85,8
2,10002,Germany,1,0.85,0.85,1
3,10002,Japan,1,0.85,0.85,1
4,10002,Spain,24,20.40,0.85,1


# RAG Pipeline

In [ ]:
# تثبيت مكتبات معالجة النصوص، التضمينات، ومخزن المتجهات
!pip install langchain langchain-community sentence-transformers chromadb rank-bm25

print("تم تثبيت مكتبات RAG بنجاح!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 66.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 76.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 86.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 

تم تثبيت مكتبات RAG بنجاح!


In [ ]:
# 1. تثبيت الحزم المطلوبة
!pip install langchain langchain-text-splitters langchain-community sentence-transformers chromadb

# 2. قراءة الملف لضمان تعريف متغير df
import pandas as pd
df = pd.read_csv('/content/online_retail.csv')

# 3. الاستيراد والتشغيل لنظام RAG
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

# تجهيز النصوص من بيانات المتجر
unique_descriptions = df['Description'].dropna().unique()[:100].tolist()
docs = [Document(page_content=desc) for desc in unique_descriptions]

# تجزئة المستندات (Document Chunking)
text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunked_docs = text_splitter.split_documents(docs)

# إنشاء نموذج التضمينات ومخزن المتجهات
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(chunked_docs, embeddings)

print(f"تم إنشاء مخزن المتجهات بنجاح ويحتوي على {len(chunked_docs)} قطعة نصية!")

/tmp/ipykernel_10324/3086796453.py:23: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

تم إنشاء مخزن المتجهات بنجاح ويحتوي على 100 قطعة نصية!


In [ ]:
# اختبار البحث في مخزن المتجهات (Vector Search)
query = "WHITE"  # كلمة بحث تجريبية عن المنتجات
results = vectorstore.similarity_search(query, k=3)

print(f"نتيجة البحث عن الاستعلام: '{query}'\n" + "-"*40)
for i, res in enumerate(results, 1):
    print(f"نتيجة {i}: {res.page_content}")

نتيجة البحث عن الاستعلام: 'WHITE'
----------------------------------------
نتيجة 1: WOODEN FRAME ANTIQUE WHITE
نتيجة 2: WHITE METAL LANTERN
نتيجة 3: WOODEN PICTURE FRAME WHITE FINISH
